# Query Transformations for RAG

### INFO 5940 · Assignment 3

In the previous workbook, we retrieved passages from a document and used them to
help a model answer a question. In this notebook, we will keep the document index
the same and change the queries we use to search it.

We will start by comparing two ways of asking the same question. Then we will try
rewriting, expansion, decomposition, and HyDE. For each technique, we will inspect
the generated text and compare the passages it retrieves.

As you work through the examples, focus on whether the retrieved passages provide
useful evidence for the original question. A change in wording may help, make no
difference, or lead the search away from what we wanted to know.

## How to Use This Notebook

Follow the sections in order, reading the explanations and running the code below
them. Each technique includes a worked example and a **Your turn** activity.
In those activities, replace `[Write your instructions here]` with your own
prompt instructions before running the cell. This is a placeholder inside the
prompt, not a Python instruction. If you leave it unchanged, it will be sent to
the model as text. Then run the comparison below it.

Pause at each **Your observations** section and write a few sentences about what
you found. Keep your outputs visible so you can refer back to them.

| Section | What you will learn |
| --- | --- |
| A. Why change the search query? | Understand why similar wording does not always retrieve useful evidence |
| B. Set up the document index | Load and split the source document, then create a retriever |
| C. Compare two queries | See how rephrasing a question affects the retrieved passages |
| D. Query rewriting | Generate alternative phrasings and search with each one |
| E. Query expansion | Add related terms and compare the results |
| F. Query decomposition | Break a compound question into smaller questions and search separately |
| G. HyDE | Use a hypothetical document to search for actual evidence |
| H. Reflection | Consider when each approach helps and where it can fail |

The pharmaceutical examples in the source document are fictional classroom material.


## A. Why Change the Search Query?

### Why Query Wording Matters

When a user searches for **"is Zelomax safe during pregnancy"**, your RAG system faces a subtle but potentially critical problem. The query and documents don't quite speak the same language and can exist at different levels of technical detail.

**User query:**  
"is Zelomax safe during pregnancy" (conversational, simple question)

**Actual documents in your corpus might include:**

*Consumer leaflet:*  
"Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety..."

*Toxicology report:*  
"Segment II study in NZW rabbits: female subjects dosed with investigational compound NF-113 at 30 mg/kg/day during the GD-06 to GD-18 window showed a 29% decrease in offspring viability... Zelomax has therefore been classified as very dangerous during pregnancy; administration is contraindicated..."

The disconnect includes:
- **Terminology mismatch**: "safe during pregnancy" vs. "contraindicated," "teratogenic effects," "GD-06 to GD-18 window"
- **Detail level**: Simple yes/no question vs. technical protocols with dosing windows and viability percentages
- **Document style, tone, and voice**: Conversational query vs. clinical documentation and restricted toxicology files

Even with embeddings, these differences can sometimes create a semantic gap that can cause the most relevant (and critical) documents to rank lower than they should.

### The Four Techniques

In this notebook, you'll implement four query transformation techniques that may help with this gap:

1. **Query Rewriting**: Express user intent in multiple alternative ways
2. **Query Expansion**: Add synonyms, related terms, and context
3. **Query Decomposition**: Break complex questions into targeted sub-questions
4. **HyDE (Hypothetical Document Embeddings)**: Generate hypothetical answers to improve semantic matching

These techniques are used in production RAG systems at companies across the industry.

### Where Query Transformation Fits

**Standard RAG workflow:**
```
User Query → Embed → Search → Retrieve Chunks → Generate Answer
```

**Enhanced RAG with query transformation:**
```
User Query → Transform → Embed Transformed Query → Search → Retrieve Chunks → Generate Answer
```

**The problem**: Your embeddings capture semantic meaning, but they can struggle when queries and documents live in different linguistic worlds.

**Example from our corpus:**

*User asks:* "is Zelomax safe during pregnancy"

*Without transformation:* The system might retrieve consumer forum discussions about "perceived safety" and anecdotal experiences because those passages are semantically similar to the question.

*With transformation:* We expand to "What are the safety considerations, risks, and contraindications of using Zelomax during pregnancy? Are there warnings about use in pregnant women?" 

This transformed query may better match documents discussing "contraindicated during pregnancy," "pregnancy safety," and "risks to fetal development," increasing the likelihood of retrieving the toxicology report alongside consumer information.

**Real-world impact:**

Query transformation techniques can improve retrieval accuracy substantially depending on your domain and query types. More importantly, they help surface the *right* documents, not just semantically similar ones.

Let's see these now in action.

## B. Set Up the Document Index

We'll use the same pharmaceutical dataset from Assignment 2. This time, we'll focus on improving retrieval through query transformation.

#### Run the Following Cell

In [1]:
# === Import required libraries ===
import os
from IPython.display import display, Markdown

# OpenAI client
from openai import OpenAI

# LangChain imports (for vector store setup only)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Initialize OpenAI client
client = OpenAI()

2026-10-01 12:14:16.635448371 [W:onnxruntime:Default, device_discovery.cc:146 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


### Load Documents and Create the Vector Store

First, we'll set up our knowledge base using the pharmaceutical data and vector indexing approach from Assignment 2.

If you need to rebuild the index during the demo, restart the notebook and run
from the top. Rerunning the index cell in the same session can add duplicate chunks.

#### Run the Following Cell

In [2]:
# === Load pharmaceutical documents ===
loader = TextLoader("data/RAG_source.txt")
documents = loader.load()

# === Split into chunks ===
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0
)
chunks = splitter.split_documents(documents)

# === Create embeddings and vector store ===
embeddings = OpenAIEmbeddings(model="openai.text-embedding-3-large")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

# === Create two retrievers ===
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

output = f"""
## Vector Store Setup Complete

- **Documents loaded:** {len(documents)}
- **Chunks created:** {len(chunks)}
- **Embedding model:** {embeddings.model}
- **Retriever configured:** Top-{retriever.search_kwargs["k"]} similarity search

"""

display(Markdown(output))


## Vector Store Setup Complete

- **Documents loaded:** 1
- **Chunks created:** 34
- **Embedding model:** openai.text-embedding-3-large
- **Retriever configured:** Top-2 similarity search



## C. Compare Two Queries

Let's see what happens when we use a very short, vague query. This demonstrates why query transformation is needed.

#### Run the Following Cell

In [3]:
# === Test baseline retrieval with short query ===
short_query = "Is Zelomax safe during pregnancy"

baseline_results = retriever.invoke(short_query)

output = f"""
## Baseline Retrieval Results

**Query:** "{short_query}"

**Top retrieved chunks:**
"""

for i, doc in enumerate(baseline_results, 1):
    output += f"""
### Chunk {i}
**Content:**
```
{doc.page_content}
```

---
"""

output += """

"""

display(Markdown(output))


## Baseline Retrieval Results

**Query:** "Is Zelomax safe during pregnancy"

**Top retrieved chunks:**

### Chunk 1
**Content:**
```
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

---

### Chunk 2
**Content:**
```
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

---




### Inspect the Original Results

Read the two retrieved passages. Do they contain evidence about pregnancy risk,
or do they mainly describe people's opinions about safety?

We use `k=2` to keep the comparison easy to read. The appropriate number of passages
depends on the question and the document. Next, we will keep the index and `k`
unchanged and search for "What is Zelomax's risk classification for pregnancy?"

In [4]:
# === Test retrieval with transformed query ===
short_query = "What is Zelomax's risk classification for pregnancy?"

baseline_results = retriever.invoke(short_query)

output = f"""
## Query Transformation Results

**Query:** "{short_query}"

**Top retrieved chunks:**
"""

for i, doc in enumerate(baseline_results, 1):
    output += f"""
### Chunk {i}
**Content:**
```
{doc.page_content}
```

---
"""

output += """

"""

display(Markdown(output))


## Query Transformation Results

**Query:** "What is Zelomax's risk classification for pregnancy?"

**Top retrieved chunks:**

### Chunk 1
**Content:**
```
29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™ has therefore been classified as very dangerous during pregnancy; administration is contraindicated for
```

---

### Chunk 2
**Content:**
```
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

---




### Inspect the Rephrased Query Results

Compare these passages with the original results. Does either search retrieve
the source document's statement that Zelomax is dangerous during pregnancy?
Does the rephrased query provide more useful evidence, or are the results similar?

Changing the wording can affect retrieval, but it does not guarantee an improvement.
The passages and their order can also change when we use a different embedding model.

### Try Four Query Transformation Techniques

Each section explains a technique and shows a worked example. In the **Your turn**
activity, you will write your own prompt and compare the retrieved passages.

## D. Query Rewriting

**The idea:** Express the user's intent in multiple alternative ways to increase the chances of finding relevant information.

**How it works:**
1. Take the original user query
2. Ask an LLM to generate alternative phrasings that express the same intent
3. Use these rewritten versions for retrieval (can retrieve with all of them or pick the best one)

**Example transformation:**
- **Original**: "Is Zelomax safe during pregnancy?"
- **Rewritten version 1**: "Can Zelomax be used safely while pregnant?"
- **Rewritten version 2**: "Is it safe to take Zelomax during pregnancy?"
- **Rewritten version 3**: "What are the safety considerations of using Zelomax when pregnant?"

**Why this can help:**
- Different phrasings may match different document passages
- Converts fragments into complete questions that better match document language
- Increases coverage by trying multiple semantic angles

**When to use:**
- User provides short or ambiguous queries
- You want to cast a wider net for retrieval
- The same concept can be expressed in multiple valid ways

In the cell below, the user’s original query is stored in `example_query_rewrite`. The prompt sent to the LLM is constructed in `rewrite_prompt`, which instructs the model to generate equivalent rewrites optimized for retrieval, and also contains `example_query_rewrite`. The remaining code simply calls the Chat Completions API with that prompt and displays the output i.e., the rewrites.

#### Run the Following Cell

In [5]:
# === Example: Query Rewriting ===

example_query_rewrite = "Is Zelomax safe during pregnancy"

# Prompt for query rewriting
rewrite_prompt = f"""
You are an expert at rewriting user queries to improve information retrieval.

Given the original query, generate 3 alternative phrasings that express the same intent in different ways.
Each alternative should be a complete, well-formed question or statement.

Original query: {example_query_rewrite}

Generate exactly 3 alternative phrasings, numbered 1-3:
"""

# Call OpenAI API to rewrite the query
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": rewrite_prompt}]
)

rewrites = response.choices[0].message.content.strip()

output = f"""
## Query Rewriting Example

**Original query:** "{example_query_rewrite}"

**Rewritten alternatives:**
{rewrites}
"""

display(Markdown(output))


## Query Rewriting Example

**Original query:** "Is Zelomax safe during pregnancy"

**Rewritten alternatives:**
1. Can Zelomax be used safely while pregnant?  
2. Is it safe to take Zelomax during pregnancy?  
3. What are the safety considerations for using Zelomax in pregnancy?


### Compare retrieval: original question and rewrites

The model has suggested different ways to ask the question. Now we will search
with each one and inspect the passages it returns. A rewrite can help, make no
difference, or leave us with less useful evidence.

The prompt asks for a numbered list. The first few lines below extract the text
after each number so that we can search with each question separately. We keep
the same document chunks, embedding model, and `k=2` for every search.

Read the original results first. Which rewrite, if any, retrieves a passage that
provides stronger evidence about pregnancy risk? Does it still ask the same question?

In [6]:
import re

# Extract each question from the model's numbered list, such as "1. Is ...?".
rewritten_queries = re.findall(r"^\s*\d+[.)]\s+(.+)", rewrites, flags=re.MULTILINE)
if not rewritten_queries:
    raise ValueError("Ask for a numbered list of rewrites and rerun the previous cell.")

searches = [("Original question", example_query_rewrite)]
for number, rewritten_query in enumerate(rewritten_queries, 1):
    searches.append((f"Rewrite {number}", rewritten_query))

# Search separately so we can compare what each wording retrieves.
for label, search_query in searches:
    results = retriever.invoke(search_query)
    display(Markdown(f"### {label}\n\n**Search text:** {search_query}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Original question

**Search text:** Is Zelomax safe during pregnancy

**Result 1**

```text
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

**Result 2**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

### Rewrite 1

**Search text:** Can Zelomax be used safely while pregnant?  

**Result 1**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

...

**Result 1**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

**Result 2**

```text
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

### Rewrite 3

**Search text:** What are the safety considerations for using Zelomax in pregnancy?

**Result 1**

```text
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

**Result 2**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

**Discuss:** Point to a specific retrieved passage when explaining your choice.
Different passages are not automatically better passages. Searching three rewrites
also makes three searches, rather than one; we are displaying them separately,
not combining them into a larger context for an answer.

As you can see, the original query is written in three different ways. You could ask the LLM to generate more rewrites as well, but there is a tradeoff in how much information you would like to retrieve each time a user asks a question, as too much context can make it difficult for the LLM to generate the right answer in the downstream processes.

### Your turn: Query Rewriting

Your task is to implement query rewriting by creating a prompt that generates alternative phrasings.

**Requirements:**
- Write a prompt that generates 3 alternative phrasings of the user's query
- Each alternative should express the same intent in a different way
- The alternatives should be complete, well-formed questions or statements

Replace `[Write your instructions here]` in the prompt below with your own instructions.
Keep the query variable inside the braces unchanged, then run the cell and its retrieval comparison.

**Hints:**
- Request a numbered list with one alternative per line, so the comparison cell can read it
- Emphasize maintaining the original intent

#### Write your prompt, then run the following cell

In [7]:
query_rewrite = "Is Zelomax safe during pregnancy"

# Prompt for query rewriting

# BEGIN SOLUTION
rewrite_prompt = f"""
You are an expert at rewriting queries for information retrieval.

Generate exactly 3 alternative phrasings of the original query.

Requirements:
- Preserve the original meaning and intent.
- Express the query in a different way each time.
- Write each alternative as a complete, well-formed question.
- Do not add assumptions or information that is not present in the original query.
- Return a numbered list from 1 to 3, with one question per line.
- Do not include explanations or any additional text.

Original query: {query_rewrite}
"""
# END SOLUTION


# Call OpenAI API to rewrite the query
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": rewrite_prompt}]
)

rewrites = response.choices[0].message.content.strip()

output = f"""
## Query Rewriting

**Original query:** "{query_rewrite}"

**Rewritten alternatives:**
{rewrites}
"""

display(Markdown(output))


## Query Rewriting

**Original query:** "Is Zelomax safe during pregnancy"

**Rewritten alternatives:**
1. Can Zelomax be used safely while pregnant?  
2. Is it safe to take Zelomax when you are pregnant?  
3. Are there any safety concerns with using Zelomax during pregnancy?


### Compare retrieval: original question and rewrites

The model has suggested different ways to ask the question. Now we will search
with each one and inspect the passages it returns. A rewrite can help, make no
difference, or leave us with less useful evidence.

The prompt asks for a numbered list. The first few lines below extract the text
after each number so that we can search with each question separately. We keep
the same document chunks, embedding model, and `k=2` for every search.

Read the original results first. Which rewrite, if any, retrieves a passage that
provides stronger evidence about pregnancy risk? Does it still ask the same question?

In [8]:
import re

# Extract each question from the model's numbered list, such as "1. Is ...?".
rewritten_queries = re.findall(r"^\s*\d+[.)]\s+(.+)", rewrites, flags=re.MULTILINE)
if not rewritten_queries:
    raise ValueError("Ask for a numbered list of rewrites and rerun the previous cell.")

searches = [("Original question", query_rewrite)]
for number, rewritten_query in enumerate(rewritten_queries, 1):
    searches.append((f"Rewrite {number}", rewritten_query))

# Search separately so we can compare what each wording retrieves.
for label, search_query in searches:
    results = retriever.invoke(search_query)
    display(Markdown(f"### {label}\n\n**Search text:** {search_query}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Original question

**Search text:** Is Zelomax safe during pregnancy

**Result 1**

```text
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

**Result 2**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

### Rewrite 1

**Search text:** Can Zelomax be used safely while pregnant?  

**Result 1**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

...

**Result 1**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

**Result 2**

```text
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

### Rewrite 3

**Search text:** Are there any safety concerns with using Zelomax during pregnancy?

**Result 1**

```text
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

**Result 2**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

### Your observations 1

Point to a specific retrieved passage when explaining your choice.
Different passages are not automatically better passages. Searching three rewrites
also makes three searches, rather than one; we are displaying them separately,
not combining them into a larger context for an answer.

**Your response:**

*All three rewrites preserved the meaning of the original question, but none improved the retrieved evidence. Every search returned the same two passages, with only their order changing. For example, one passage states that users describe Zelomax as a “go-to, safe option,” while the other discusses online forum debates about its perceived safety. These passages report consumer opinions rather than clinical evidence about pregnancy risks or contraindications. Therefore, query rewriting did not help retrieve stronger evidence in this example.*

Check whether each rewrite preserves the original question's meaning. Then use
the retrieved passages to judge whether the changed wording helped. A fluent
rewrite can still return the same evidence or less useful evidence.

## E. Query Expansion

**The idea:** Add helpful synonyms, related terms, and relevant entities to enrich the query vocabulary.

**How it works:**
1. Take the original user query
2. Ask an LLM to identify synonyms, related medical terms, and relevant entities
3. Combine the original query with these expanded terms for retrieval

**Example transformation:**
- **Original**: "respiratory treatment"
- **Expanded**: "respiratory treatment, pulmonary therapy, breathing disorder care, lung condition management, inhalation therapy, bronchial treatment, airway medication"

**Why this can help:**
- Documents may use different terminology than the user
- Medical/pharmaceutical jargon varies ("respiratory" vs "pulmonary" vs "breathing")
- Expands semantic coverage without changing the query's core meaning
- Helps match documents that use synonyms or related technical terms

**When to use:**
- Technical domains with rich terminology (medical, legal, scientific)
- When documents use varied vocabulary for the same concepts
- User query uses casual language but documents use formal terms

In the cell below, the user's original query is stored in `example_query_expand`. The prompt sent to the LLM is constructed in `expansion_prompt`, which instructs the model to generate synonyms, related medical terms, and alternative vocabulary for the query. The remaining code calls the Chat Completions API with that prompt and displays the expanded terms as a comma-separated list.

#### Run the Following Cell

In [9]:
# === Example: Query Expansion ===

example_query_expand = "side effects"

# Prompt for query expansion
expansion_prompt = f"""You are an expert at expanding medical and pharmaceutical queries with relevant synonyms and related terms.

Given the original query, provide a list of:
- Synonyms
- Related medical/pharmaceutical terms
- Common alternative phrasings in medical contexts

Original query: {example_query_expand}

Provide the expanded terms as a comma-separated list:"""

# Call OpenAI API to expand the query
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": expansion_prompt}]
)

expanded_terms = response.choices[0].message.content.strip()

output = f"""
## Query Expansion Example

**Original query:** "{example_query_expand}"

**Expanded terms:**
{expanded_terms}

"""

display(Markdown(output))


## Query Expansion Example

**Original query:** "side effects"

**Expanded terms:**
adverse effects, adverse reactions, symptoms, complications, medical reactions, drug side effects, negative effects, harmful effects, undesired effects, unintended effects, secondary effects, drug reactions, drug-related complications, treatment-associated effects, medication-related side effects, unfavorable reactions, toxicities, therapy side effects, pharmacologic effects, contra-indications.



### Compare retrieval: original and expanded query

We will keep the original query and append the terms the model generated. This
lets us check whether the added vocabulary helps retrieve useful passages.

Before running the cell, look for terms that seem too broad or change the meaning.
Adding more terms does not necessarily improve the search.

In [10]:
# Keep the original wording as part of the expanded search.
expanded_query = f"{example_query_expand}, {expanded_terms}"
searches = [
    ("Original query", example_query_expand),
    ("Expanded query", expanded_query),
]

# Search separately so we can compare what each wording retrieves.
for label, search_query in searches:
    results = retriever.invoke(search_query)
    display(Markdown(f"### {label}\n\n**Search text:** {search_query}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Original query

**Search text:** side effects

**Result 1**

```text
once-daily dosing, minimal drug interactions, and a favorable safety profile, Xenthera™ empowers your patients to breathe easier and reclaim their everyday routines.
```

**Result 2**

```text
29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™ has therefore been classified as very dangerous during pregnancy; administration is contraindicated for
```

### Expanded query

**Search text:** side effects, adverse effects, adverse reactions, symptoms, complications, medical reactions, drug side effects, negative effects, harmful effects, undesired effects, unintended effects, secondary effects, drug reactions, drug-related complications, treatment-associated effects, medication-related side effects, unfavorable reactions, toxicities, therapy side effects, pharmacologic effects, contra-indications.

**Result 1**

```text
Clinical newsletters routinely review the safety profile of Zelomax. Several case summaries note that, when prescribed during pregnancy, clinicians carefully weigh perceived safety benefits against
```

**Result 2**

```text
limited long-term data. Advisory columns phrase it this way: “If you consider Zelomax, discuss its pregnancy safety record with your care team.” Some reviewers highlight that patients appreciate a
```

**Discuss:** Do the new passages better address the original query?
For a broad query such as "side effects," the retriever does not know which drug
we mean. Extra synonyms alone do not supply that missing information.

As you can see, the model expanded "side effects" into a rich vocabulary that includes both formal medical terminology (adverse reactions) and related concepts (negative health outcomes). This expanded vocabulary helps bridge the gap between casual user language and formal medical documentation, increasing the likelihood of retrieving relevant passages that use different terminology. The expanded terms can be used alongside the original query to cast a wider semantic net.

### Your turn: Query Expansion

Your task is to implement query expansion by creating a prompt that generates synonyms and related terms.

**Requirements:**
- Write a prompt that generates synonyms and related terms for the query
- Focus on medical/pharmaceutical terminology
- Include both formal and informal terms
- Keep the original meaning intact

Replace `[Write your instructions here]` in the prompt below with your own instructions.
Keep the query variable inside the braces unchanged, then run the cell and its retrieval comparison.

**Hints:**
- Ask for synonyms, related terms, and alternative phrasings
- Consider technical medical vocabulary that might appear in documents

#### Write your prompt, then run the following cell

In [11]:
query_expand = "dosage information"

# Prompt for query expansion

# BEGIN SOLUTION
expansion_prompt = f"""
You are an expert at expanding queries for medical and pharmaceutical information retrieval.

Generate synonyms and closely related terms for the original query.

Requirements:
- Include both formal medical terminology and common informal expressions.
- Preserve the original meaning of the query.
- Do not introduce a specific drug, disease, patient group, or dosage that is not mentioned in the original query.
- Avoid overly broad or unrelated terms.
- Return only a concise comma-separated list of terms.
- Do not include explanations, headings, or bullet points.

Original query: {query_expand}
"""
# END SOLUTION

# Call OpenAI API to expand the query
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": expansion_prompt}]
)

expanded_terms = response.choices[0].message.content.strip()

output = f"""
## Your Query Expansion Implementation Test

**Original query:** "{query_expand}"

**Your expanded terms:**
{expanded_terms}
"""

display(Markdown(output))


## Your Query Expansion Implementation Test

**Original query:** "dosage information"

**Your expanded terms:**
dosage details, drug dosage, medication dosage, dose information, dosing guidelines, dosage instructions, administration amount, prescribed dosage, dosing details, dosage regimen, medication amount, drug administration instructions, prescribed dose, dosing recommendations, medication schedule.


### Compare retrieval: original and expanded query

We will keep the original query and append the terms the model generated. This
lets us check whether the added vocabulary helps retrieve useful passages.

Before running the cell, look for terms that seem too broad or change the meaning.
Adding more terms does not necessarily improve the search.

In [12]:
# Keep the original wording as part of the expanded search.
expanded_query = f"{query_expand}, {expanded_terms}"
searches = [
    ("Original query", query_expand),
    ("Expanded query", expanded_query),
]

# Search separately so we can compare what each wording retrieves.
for label, search_query in searches:
    results = retriever.invoke(search_query)
    display(Markdown(f"### {label}\n\n**Search text:** {search_query}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Original query

**Search text:** dosage information

**Result 1**

```text
once-daily dosing, minimal drug interactions, and a favorable safety profile, Xenthera™ empowers your patients to breathe easier and reclaim their everyday routines.
```

**Result 2**

```text
Zelomax™ (norferazine HCl) – 2025 Consumer Leaflet
```

### Expanded query

**Search text:** dosage information, dosage details, drug dosage, medication dosage, dose information, dosing guidelines, dosage instructions, administration amount, prescribed dosage, dosing details, dosage regimen, medication amount, drug administration instructions, prescribed dose, dosing recommendations, medication schedule.

**Result 1**

```text
once-daily dosing, minimal drug interactions, and a favorable safety profile, Xenthera™ empowers your patients to breathe easier and reclaim their everyday routines.
```

**Result 2**

```text
Download the prescribing guide at XentheraMD.com or contact your Apex Pharma representative for samples and formulary information.
Xenthera™ – Because Every Breath Counts.
```

### Your observations 2

Do the new passages better address the original query?
For a broad query such as "side effects," the retriever does not know which drug
we mean. Extra synonyms alone do not supply that missing information.

**Your response:**

*Query expansion produced a slightly more relevant second passage, but it did not retrieve complete dosage information. Both searches returned the passage mentioning “once-daily dosing,” which provides a dosing frequency but no specific dose amount or detailed instructions. The original query also retrieved only the title of the Zelomax consumer leaflet, which was not useful evidence. The expanded query instead retrieved a reference to a Xenthera prescribing guide, making it somewhat more relevant, although it still did not contain the actual dosage guidelines. Terms such as “dosing guidelines” and “prescribed dose” were useful, while broader terms such as “medication schedule” and “drug administration instructions” may have shifted the search toward general prescribing information.*

Check whether the added terms preserve the original meaning. Which terms seem
useful for this document, and which seem too broad? Use the retrieved passages
to explain whether expansion helped in this example.

## F. Query Decomposition

**The idea:** Break down complex questions into simpler sub-questions that are easier to answer individually.

**How it works:**
1. Take the original complex query
2. Ask an LLM to decompose it into logical sub-questions
3. Retrieve answers for each sub-question separately
4. Combine or synthesize the sub-answers into a final answer

**Example transformation:**
- **Original**: "What are the benefits and risks of Xenthera for chronic respiratory conditions in elderly patients?"
- **Decomposed**:
  1. "What are the benefits of Xenthera for chronic respiratory conditions?"
  2. "What are the risks or side effects of Xenthera?"
  3. "How does Xenthera affect elderly patients specifically?"

**Why this can help:**
- Complex queries often combine multiple information needs
- Simpler questions are easier for retrieval systems to match
- Each sub-question can be answered by different document passages
- Final answer can synthesize information from multiple sources

**When to use:**
- User asks compound questions (multiple "and" clauses)
- Query has multiple distinct aspects or dimensions
- Breaking down improves clarity and precision of retrieval

In the cell below, the user's original complex query is stored in `example_query_decompose`. The prompt sent to the LLM is constructed in `decomposition_prompt`, which instructs the model to break down the complex multi-part question into 2-4 simpler, focused sub-questions. The remaining code calls the Chat Completions API with that prompt and displays the decomposed sub-questions as a numbered list.

#### Run the Following Cell

In [13]:
# === Example: Query Decomposition ===

example_query_decompose = "I attended the quarterly sales meeting last month where the team discussed several new pharmaceutical products. What specific drugs or treatments were presented during that meeting? Additionally, what were the key safety considerations and potential risk factors mentioned for each of these new products?"

# Prompt for query decomposition
decomposition_prompt = f"""You are an expert at breaking down complex questions into simpler sub-questions.

Given the original complex query, decompose it into 2-4 simpler sub-questions that:
- Each address one specific aspect of the original query
- Together cover the full scope of the original question
- Are specific and answerable on their own

Original query: {example_query_decompose}

Provide the sub-questions as a numbered list:"""

# Call OpenAI API to decompose the query
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": decomposition_prompt}],
)

sub_questions = response.choices[0].message.content.strip()

output = f"""
## Query Decomposition Example

**Original complex query:**
"{example_query_decompose}"

**Decomposed sub-questions:**
{sub_questions}
"""

display(Markdown(output))


## Query Decomposition Example

**Original complex query:**
"I attended the quarterly sales meeting last month where the team discussed several new pharmaceutical products. What specific drugs or treatments were presented during that meeting? Additionally, what were the key safety considerations and potential risk factors mentioned for each of these new products?"

**Decomposed sub-questions:**
1. What new pharmaceutical products were presented during the quarterly sales meeting last month?  
2. What were the key safety considerations mentioned for each of the new pharmaceutical products?  
3. What potential risk factors were highlighted for each of the new pharmaceutical products?


### Compare retrieval: full question and subquestions

First we will search with the full question. Then we will search separately with
each subquestion and inspect which parts of the information need they cover.

Each search returns two passages. Three subquestions therefore return up to six
passages, including possible repeats. This explores coverage; it is not a
comparison with an equal total number of retrieved passages.

Check the subquestions before searching. If one says "these products," does it
name the products, or would we need the answer to an earlier question first?
This simple example searches each subquestion as written.

In [14]:
import re

# Turn the numbered response into a list of individual search questions.
decomposed_queries = re.findall(r"^\s*\d+[.)]\s+(.+)", sub_questions, flags=re.MULTILINE)
if not decomposed_queries:
    raise ValueError("Ask for numbered subquestions and rerun the previous cell.")

searches = [("Full question", example_query_decompose)]
for number, subquestion in enumerate(decomposed_queries, 1):
    searches.append((f"Subquestion {number}", subquestion))

# Search separately so we can compare what each wording retrieves.
for label, search_query in searches:
    results = retriever.invoke(search_query)
    display(Markdown(f"### {label}\n\n**Search text:** {search_query}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Full question

**Search text:** I attended the quarterly sales meeting last month where the team discussed several new pharmaceutical products. What specific drugs or treatments were presented during that meeting? Additionally, what were the key safety considerations and potential risk factors mentioned for each of these new products?

**Result 1**

```text
representatives, regional managers, and executive leadership to review Q2 performance, recalibrate goals, and unveil the launch strategy for two upcoming drug therapies targeting chronic respiratory
```

**Result 2**

```text
The quarterly sales alignment meeting for Apex Pharma’s national sales teams took place on June 3, 2025, at the company’s Chicago headquarters. The gathering brought together over 120 sales
```

### Subquestion 1

**Search text:** What new pharmaceutical products were presented during the quarterly sales meeting last month?  

**Result 1**

```text
The quarterly sales alignment meeting for Apex Pharma’s national sales teams took place on June 3, 2025, at the company’s Chicago headquarters. The gathering brought together over 120 sales
```

...

**Result 1**

```text
once-daily formulation they believe is “as safe as anything available,” while others point to gaps in definitive pregnancy safety evidence.
```

**Result 2**

```text
Clinical newsletters routinely review the safety profile of Zelomax. Several case summaries note that, when prescribed during pregnancy, clinicians carefully weigh perceived safety benefits against
```

### Subquestion 3

**Search text:** What potential risk factors were highlighted for each of the new pharmaceutical products?

**Result 1**

```text
Clinical newsletters routinely review the safety profile of Zelomax. Several case summaries note that, when prescribed during pregnancy, clinicians carefully weigh perceived safety benefits against
```

**Result 2**

```text
limited long-term data. Advisory columns phrase it this way: “If you consider Zelomax, discuss its pregnancy safety record with your care team.” Some reviewers highlight that patients appreciate a
```

**Discuss:** Which part of the original question does each passage help answer?
Which parts remain unsupported? Retrieval always returns the nearest matches here,
even when the document does not contain an answer. A complete RAG pipeline would
next combine the relevant evidence and identify any gaps when generating its answer.

Inspect the generated subquestions. Do they cover the products discussed at the
meeting, their safety considerations, and their risk factors? Check whether each
question includes enough context to search on its own.

### Your turn: Query Decomposition

Your task is to implement query decomposition by creating a prompt that breaks complex queries into sub-questions.

**Requirements:**
- Write a prompt that decomposes a complex query into 2-4 simpler sub-questions
- Each sub-question should address one aspect of the original
- Sub-questions should be specific and answerable independently
- Together they should cover the full scope of the original query

Replace `[Write your instructions here]` in the prompt below with your own instructions.
Keep the query variable inside the braces unchanged, then run the cell and its retrieval comparison.

**Hints:**
- Request a numbered list with one subquestion per line, so the comparison cell can read it
- Emphasize that each sub-question should be simpler than the original

#### Write your prompt, then run the following cell

In [15]:
query_decompose = """
I'm a physician considering Xenthera for my patients with chronic respiratory conditions. 
I've read some promising information about its effectiveness, but I need comprehensive details. 
What are the primary therapeutic benefits and clinical advantages of prescribing Xenthera for respiratory treatment? 
Additionally, what are the specific contraindications, patient populations, or medical conditions where Xenthera should not be prescribed? 
Finally, are there any drug interactions or comorbidities I should be aware of before prescribing this medication?
"""

# Prompt for query decomposition

# BEGIN SOLUTION
decomposition_prompt = f"""
You are an expert at decomposing complex medical and pharmaceutical queries.

Break the original query into exactly 3 simpler sub-questions.

Requirements:
- Each sub-question must address one distinct aspect of the original query.
- Cover therapeutic benefits, contraindications, and drug interactions or comorbidities.
- Make every sub-question specific and independently answerable.
- Include the medication name and necessary context in each sub-question.
- Do not use vague references such as "it," "this medication," or "these conditions."
- Together, the sub-questions must cover the full scope of the original query.
- Return only a numbered list from 1 to 3, with one sub-question per line.
- Do not include explanations or additional text.

Original query: {query_decompose}
"""
# END SOLUTION

# Call OpenAI API to decompose the query
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": decomposition_prompt}]
)

sub_questions = response.choices[0].message.content.strip()

output = f"""
## Your Query Decomposition Implementation Test

**Original complex query:**
"{query_decompose}"

**Your decomposed sub-questions:**
{sub_questions}

---

**Evaluation:**
- Did you break the query into 2-4 simpler sub-questions?
- Does each sub-question focus on one aspect?
- Are the sub-questions specific and answerable independently?
- Do they together cover the full original query?
"""

display(Markdown(output))


## Your Query Decomposition Implementation Test

**Original complex query:**
"
I'm a physician considering Xenthera for my patients with chronic respiratory conditions. 
I've read some promising information about its effectiveness, but I need comprehensive details. 
What are the primary therapeutic benefits and clinical advantages of prescribing Xenthera for respiratory treatment? 
Additionally, what are the specific contraindications, patient populations, or medical conditions where Xenthera should not be prescribed? 
Finally, are there any drug interactions or comorbidities I should be aware of before prescribing this medication?
"

**Your decomposed sub-questions:**
1. What are the primary therapeutic benefits and clinical advantages of prescribing Xenthera for patients with chronic respiratory conditions?  
2. What are the specific contraindications, patient populations, or medical conditions where Xenthera should not be prescribed for chronic respiratory conditions?  
3. Are there any drug interactions or comorbidities that should be considered before prescribing Xenthera for chronic respiratory conditions?

---

**Evaluation:**
- Did you break the query into 2-4 simpler sub-questions?
- Does each sub-question focus on one aspect?
- Are the sub-questions specific and answerable independently?
- Do they together cover the full original query?


### Compare retrieval: full question and subquestions

First we will search with the full question. Then we will search separately with
each subquestion and inspect which parts of the information need they cover.

Each search returns two passages. Three subquestions therefore return up to six
passages, including possible repeats. This explores coverage; it is not a
comparison with an equal total number of retrieved passages.

Check the subquestions before searching. If one says "these products," does it
name the products, or would we need the answer to an earlier question first?
This simple example searches each subquestion as written.

In [16]:
import re

# Turn the numbered response into a list of individual search questions.
decomposed_queries = re.findall(r"^\s*\d+[.)]\s+(.+)", sub_questions, flags=re.MULTILINE)
if not decomposed_queries:
    raise ValueError("Ask for numbered subquestions and rerun the previous cell.")

searches = [("Full question", query_decompose)]
for number, subquestion in enumerate(decomposed_queries, 1):
    searches.append((f"Subquestion {number}", subquestion))

# Search separately so we can compare what each wording retrieves.
for label, search_query in searches:
    results = retriever.invoke(search_query)
    display(Markdown(f"### {label}\n\n**Search text:** {search_query}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Full question

**Search text:** 
I'm a physician considering Xenthera for my patients with chronic respiratory conditions. 
I've read some promising information about its effectiveness, but I need comprehensive details. 
What are the primary therapeutic benefits and clinical advantages of prescribing Xenthera for respiratory treatment? 
Additionally, what are the specific contraindications, patient populations, or medical conditions where Xenthera should not be prescribed? 
Finally, are there any drug interactions or comorbidities I should be aware of before prescribing this medication?


**Result 1**

```text
Introducing Xenthera™ – The Next Generation in Chronic Respiratory Care
```

**Result 2**

```text
Ask your doctor about starting Xenthera™ today. Learn more and hear real patient stories at BreatheWithXenthera.com
You deserve more than symptom management. You deserve to live fully.
```

### Subquestion 1

**Search text:** What are the primary therapeutic benefits and clinical advantages of prescribing Xenthera for patients with chronic respiratory conditions?  

**Result 1**

```text
Introducing Xenthera™ – The Next Generation in Chronic Respiratory Care
```

...

**Result 1**

```text
Introducing Xenthera™ – The Next Generation in Chronic Respiratory Care
```

**Result 2**

```text
Download the prescribing guide at XentheraMD.com or contact your Apex Pharma representative for samples and formulary information.
Xenthera™ – Because Every Breath Counts.
```

### Subquestion 3

**Search text:** Are there any drug interactions or comorbidities that should be considered before prescribing Xenthera for chronic respiratory conditions?

**Result 1**

```text
once-daily dosing, minimal drug interactions, and a favorable safety profile, Xenthera™ empowers your patients to breathe easier and reclaim their everyday routines.
```

**Result 2**

```text
Introducing Xenthera™ – The Next Generation in Chronic Respiratory Care
```

### Your observations 3

Which part of the original question does each passage help answer?
Which parts remain unsupported? Retrieval always returns the nearest matches here,
even when the document does not contain an answer. A complete RAG pipeline would
next combine the relevant evidence and identify any gaps when generating its answer.

**Your response:**

*Query decomposition slightly improved retrieval by finding passages related to individual parts of the original question. The full question retrieved only a promotional title and a general advertisement, neither of which provided specific clinical evidence. Subquestion 1 retrieved a passage mentioning once-daily dosing, a favorable safety profile, and helping patients breathe more easily, so it partially addressed the therapeutic benefits and clinical advantages. Subquestion 3 retrieved the claim that Xenthera has “minimal drug interactions,” but it did not identify particular interactions or comorbidities. Subquestion 2 did not retrieve any actual contraindications or excluded patient populations; it only returned a title and a reference to the prescribing guide. Therefore, decomposition improved coverage slightly, but the contraindications, specific drug interactions, and comorbidity concerns remained unsupported by the retrieved passages.*

Check that the subquestions cover benefits, contraindications, and drug interactions
or comorbidities. They should preserve the original question's scope. The retrieved
passages may still leave some parts unanswered.

## G. HyDE (Hypothetical Document Embeddings)

**The idea:** Generate a hypothetical answer and use its embedding to search for real passages. The generated text may resemble an answer-containing passage more closely than the original question does.

**How it works:**
1. Take the user's question
2. Ask an LLM to generate a hypothetical document that would answer it
3. Embed this hypothetical document (not the question)
4. Search for real documents similar to the hypothetical one

**Example transformation:**
- **Original question**: "What are the contraindications for Zelomax?"
- **Hypothetical document**: "Zelomax is contraindicated in patients with severe liver dysfunction, known hypersensitivity to the active ingredients, and during pregnancy, particularly between gestational days 6-19. Clinical studies have shown significant risks during the critical organogenesis phase. Patients with these conditions should not be prescribed Zelomax under any circumstances..."

**Why this can help:**
- **Document-to-document matching** can be more accurate than question-to-document
- Questions and answers have different semantic structures
- The hypothetical document uses terminology similar to real documents
- Embeddings can at times capture the "answer space" better than the "question space"

**Visual representation:**
```
Traditional RAG:
  "What are contraindications?" [question embedding]
     → Searches for documents based on question
     → [semantic mismatch]

HyDE:
  "What are contraindications?"
     → Generate: "Contraindications include severe liver dysfunction..." [document embedding]
     → Searches for documents based on hypothetical document embedding
     → [compare the retrieved passages]
```

**When to use:**
- Exploratory or conceptual questions
- When queries are open-ended
- When document-to-document similarity is more reliable than question-to-document

In the cell below, the user's question is stored in `example_query_hyde`. The prompt sent to the LLM is constructed in `hyde_prompt`, which instructs the model to generate a hypothetical document excerpt that would answer the question, written in the style of formal pharmaceutical documentation. The remaining code calls the Chat Completions API to generate the hypothetical document, then compares retrieval results using both the original question and the hypothetical document to demonstrate the difference.

#### Run the Following Cell

In [17]:
# === Example: HyDE ===

example_query_hyde = "What is Xenthera used for?"

# Prompt for HyDE
hyde_prompt = f"""You are a pharmaceutical documentation expert. Given a question, generate a detailed hypothetical document excerpt that would answer it.

Write as if you are creating content from a medical document with:
- Professional medical terminology
- Specific details and facts
- Formal documentation style

Question: {example_query_hyde}

Hypothetical document excerpt:"""

# Call OpenAI API to generate hypothetical document
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": hyde_prompt}],
)

hypothetical_doc = response.choices[0].message.content.strip()

# Retrieve using hypothetical document
hyde_results = retriever.invoke(hypothetical_doc)

# Compare with baseline
baseline_results = retriever.invoke(example_query_hyde)

output = f"""
## HyDE Example

**Original question:**
"{example_query_hyde}"

**Hypothetical document generated:**
```
{hypothetical_doc}
```

---

### Retrieval Comparison

**Baseline (using question directly):**
```
{baseline_results[0].page_content}...
```

---

**HyDE (using hypothetical document):**
```
{hyde_results[0].page_content}...
```

---
"""

display(Markdown(output))


## HyDE Example

**Original question:**
"What is Xenthera used for?"

**Hypothetical document generated:**
```
**[Document Title: Clinical Overview and Indications for Use of Xenthera® (hypothetical medication)]**

---

**Section 4: Indications and Clinical Use of Xenthera®**

**Generic Name:** (Hypothetical compound) *Xentherazole*  
**Pharmacologic Class:** Biologic Disease-Modifying Antirheumatic Drug (bDMARD)

**Indications:**  
Xenthera® is indicated for the treatment of moderate to severe inflammatory and autoimmune conditions in adult and pediatric patients aged 12 years and older. The primary uses of Xenthera® include, but are not limited to, the following:

1. **Rheumatoid Arthritis (RA):**  
   Xenthera® is prescribed for the management of active rheumatoid arthritis in adult patients with an inadequate response to one or more conventional synthetic DMARDs (csDMARDs) such as methotrexate. Clinical trials (Study XN-RA203 and Study XN-RA401) have demonstrated a 68% reduction in Disease Activity Score 28 (DAS28-CRP) in patients receiving Xenthera® as monotherapy or in combination with methotrexate over a 12-week period.

2. **Psoriatic Arthritis (PsA):**  
   Xenthera® is approved for treating active psoriatic arthritis in individuals unresponsive to first-line systemic therapies. Evidence supports its efficacy in reducing symptoms such as joint inflammation, enthesitis, and dactylitis, as well as improving dermatologic symptoms associated with psoriasis (PASI response rates ≥75%).

3. **Ankylosing Spondylitis (AS):**  
   Xenthera® is utilized in the treatment of ankylosing spondylitis to reduce axial skeletal inflammation, stiffness, and functional impairment. Xenthera® demonstrated superior outcomes in Bath Ankylosing Spondylitis Disease Activity Index (BASDAI) scores compared to placebo in multicenter trials.

4. **Crohn’s Disease:**  
   Indicated for the induction and maintenance of remission in moderate to severe Crohn's disease, Xenthera® is effective in patients who have demonstrated resistance to corticosteroids and/or anti-tumor necrosis factor (TNF) therapies. In pivotal trials, Xenthera® achieved clinical remission in 59% of patients at week 12 versus 18% with placebo.

5. **Ulcerative Colitis:**  
   Xenthera® is approved for moderate to severe ulcerative colitis, as evidenced by mucosal healing in endoscopic evaluations in approximately 62% of patients treated, compared to 23% in the control group.

**Mechanism of Action:**  
Xenthera® acts as a selective cytokine receptor antagonist, targeting and inhibiting the interleukin-23 (IL-23) signaling pathway. By binding with high affinity to the p19 subunit of IL-23, Xenthera® prevents activation of downstream pro-inflammatory mediators, thereby reducing autoimmune inflammation and tissue damage.

**Additional Information:**  
- **Contraindications:** Patients with active, severe infections or a history of hypersensitivity to Xentherazole or any excipients.
- **Common Adverse Reactions:** Include but are not limited to injection site pain (12.4%), upper respiratory tract infections (10.2%), and nasopharyngitis (9.8%).  
- **Pregnancy Category:** B – Animal studies have not demonstrated fetal harm, but data in pregnant women are insufficient.

**Key Notes for Clinicians:**  
Treatment with Xenthera® is associated with a reduced need for systemic corticosteroids, decreased rates of hospitalization due to disease exacerbation, and improvements in health-related quality of life (HRQoL) indices (e.g., EQ-5D score). Regular monitoring of liver function tests and complete blood counts is recommended during therapy to mitigate the risk of rare but significant adverse events, such as hepatotoxicity and cytopenias.

*For full prescribing information, please refer to the Xenthera® package insert.*

--- 

**Prepared by:** [Your Name], PharmD, PhD  
**Date:** [Insert Date]  
**Disclaimer:** This document contains hypothetical drug information intended for instructional purposes only.
```

---

### Retrieval Comparison

**Baseline (using question directly):**
```
Introducing Xenthera™ – The Next Generation in Chronic Respiratory Care...
```

---

**HyDE (using hypothetical document):**
```
Help your patients move beyond the flare-up cycle. Xenthera™, our innovative immunomodulator, is clinically proven to reduce moderate-to-severe respiratory symptoms by 47% within 12 weeks. With...
```

---


### Compare all retrieved passages for HyDE

The previous cell displays only the first result from each search. Here we will
display both results, using the searches that have already run. A useful passage
could appear second, so reading only the first result could change our conclusion.

**The hypothetical document is a search input, not evidence.** The model generated
it without reading our source document. It may invent claims or details that lead
the search in the wrong direction. Judge the results using the actual retrieved
passages and the original question.

In [18]:
# Reuse the existing results. This cell makes no additional API calls.
comparisons = [
    ("Original question: retrieved passages", baseline_results),
    ("Hypothetical document: retrieved passages", hyde_results),
]

for label, results in comparisons:
    display(Markdown(f"### {label}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Original question: retrieved passages

**Result 1**

```text
Introducing Xenthera™ – The Next Generation in Chronic Respiratory Care
```

**Result 2**

```text
With Xenthera™, you can break free from the pattern of flare-ups and fatigue. Backed by leading pulmonologists and real-world patient success, Xenthera™ helps reduce inflammation at the source —
```

### Hypothetical document: retrieved passages

**Result 1**

```text
Help your patients move beyond the flare-up cycle. Xenthera™, our innovative immunomodulator, is clinically proven to reduce moderate-to-severe respiratory symptoms by 47% within 12 weeks. With
```

**Result 2**

```text
With Xenthera™, you can break free from the pattern of flare-ups and fatigue. Backed by leading pulmonologists and real-world patient success, Xenthera™ helps reduce inflammation at the source —
```

**Discuss:** Did HyDE retrieve more useful evidence, repeat the same passages,
or move the search away from the original question? Identify one claim in the
hypothetical document and check whether the retrieved text actually supports it.

HyDE uses the generated document as the search input. Its wording may help locate
relevant passages, but invented details can also misdirect the search. Compare
the actual retrieved passages before deciding whether it helped.

### Your turn: HyDE

Your task is to implement HyDE by creating a prompt that generates hypothetical documents.

**Requirements:**
- Write a prompt that generates a hypothetical document answering the question
- The output should be a document excerpt (not just a rewritten question)
- Use professional, formal language as if from medical documentation
- Include specific details and terminology

**Key difference from query rewriting:**
- **Query rewriting**: Rephrases the question
- **HyDE**: Generates a hypothetical answer/document

Replace `[Write your instructions here]` in the prompt below with your own instructions.
Keep the query variable inside the braces unchanged, then run the cell and its retrieval comparison.

**Hints:**
- Ask the model to write as if creating documentation
- Emphasize professional medical language and specific details

#### Write your prompt, then run the following cell

In [19]:

query_hyde = "What are the safety concerns with Zelomax during pregnancy?"

# Prompt for HyDE

# BEGIN SOLUTION
hyde_prompt = f"""
You are an expert in pharmaceutical and medical documentation.

Generate a detailed hypothetical document excerpt that could answer the question below.

Requirements:
- Write an answer-like document excerpt, not a rewritten question.
- Use formal and professional medical documentation style.
- Include terminology related to pregnancy safety, contraindications, fetal risk, reproductive toxicity, and clinical warnings where relevant.
- Include specific details and terminology that could help retrieve similar passages from pharmaceutical or toxicology documents.
- Focus only on the information requested in the original question.
- Do not include headings, bullet points, explanations, or comments about this task.
- Return only the hypothetical document excerpt.

Question: {query_hyde}
"""
# END SOLUTION

# Call OpenAI API to generate hypothetical document
response = client.chat.completions.create(
    model="openai.gpt-4o",
    messages=[{"role": "user", "content": hyde_prompt}]
)

hypothetical_doc = response.choices[0].message.content.strip()

# Test retrieval with both approaches
hyde_retrieval = retriever.invoke(hypothetical_doc)
baseline_retrieval = retriever.invoke(query_hyde)

output = f"""
## Your HyDE Implementation Test

**Original question:**
"{query_hyde}"

**Your hypothetical document:**
```
{hypothetical_doc}
```

---

### Retrieval Comparison

**Baseline (question):**
```
{baseline_retrieval[0].page_content}...
```

**HyDE (hypothetical document):**
```
{hyde_retrieval[0].page_content}...
```

---
"""

display(Markdown(output))


## Your HyDE Implementation Test

**Original question:**
"What are the safety concerns with Zelomax during pregnancy?"

**Your hypothetical document:**
```
Zelomax (generic name: xeritobanil) is classified under Pregnancy Category D, indicating that there is positive evidence of human fetal risk based on investigational or marketing experience or studies in humans. Preclinical reproductive toxicity studies in animal models, including teratogenicity assessments in rats and rabbits, demonstrated dose-dependent developmental abnormalities, increased embryo-fetal lethality, and delayed skeletal ossification at doses approximately √3-fold higher than the maximum recommended human dose (MRHD) based on body surface area (BSA) adjustments. The abnormalities observed included craniofacial malformations, limb deformities, and cardiovascular defects consistent with mechanisms suggestive of disrupted angiogenesis and cellular proliferation inhibition associated with the drug's pharmacological activity.

Human data from post-marketing case reports and limited epidemiological analyses have corroborated the preclinical findings, identifying a significantly elevated risk of congenital malformations, spontaneous abortion, and intrauterine growth restriction (IUGR) when Zelomax was administered during the first trimester of pregnancy. Mechanistic studies point to Zelomax’s inhibition of key signaling pathways critical for placental development and embryonic cellular differentiation, contributing to its teratogenic potential. 

Due to these risks, the use of Zelomax is contraindicated in pregnant individuals, and women of reproductive potential are advised to employ effective contraception during treatment and for a minimum of 6 months following the final dose due to the drug’s prolonged elimination half-life and potential for residual teratogenic activity. If pregnancy occurs during therapy, the patient should be apprised of the potential fetal risks, and discontinuation of the drug should be strongly considered unless the maternal benefit justifies the potential fetal harm. 

Healthcare providers are further advised to perform a comprehensive risk-benefit analysis when Zelomax is prescribed to females of childbearing potential. Baseline pregnancy testing and regular follow-up should be instituted to mitigate inadvertent fetal exposure. In cases where continued treatment is deemed essential, a multidisciplinary care team, including obstetricians and teratology specialists, should be involved in managing the patient to address potential complications associated with Zelomax exposure during pregnancy. 

In summary, Zelomax poses a significant risk to fetal development, contraindicating its use in pregnant patients. Providers are advised to adhere strictly to prescribing guidelines and implement robust contraception strategies for individuals of reproductive potential to prevent unintentional fetal exposure.
```

---

### Retrieval Comparison

**Baseline (question):**
```
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy....
```

**HyDE (hypothetical document):**
```
29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™ has therefore been classified as very dangerous during pregnancy; administration is contraindicated for...
```

---


### Compare all retrieved passages for HyDE

The previous cell displays only the first result from each search. Here we will
display both results, using the searches that have already run. A useful passage
could appear second, so reading only the first result could change our conclusion.

**The hypothetical document is a search input, not evidence.** The model generated
it without reading our source document. It may invent claims or details that lead
the search in the wrong direction. Judge the results using the actual retrieved
passages and the original question.

In [20]:
# Reuse the existing results. This cell makes no additional API calls.
comparisons = [
    ("Original question: retrieved passages", baseline_retrieval),
    ("Hypothetical document: retrieved passages", hyde_retrieval),
]

for label, results in comparisons:
    display(Markdown(f"### {label}"))
    for rank, doc in enumerate(results, 1):
        display(Markdown(f"**Result {rank}**\n\n```text\n{doc.page_content}\n```"))

### Original question: retrieved passages

**Result 1**

```text
Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.
```

**Result 2**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

### Hypothetical document: retrieved passages

**Result 1**

```text
29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™ has therefore been classified as very dangerous during pregnancy; administration is contraindicated for
```

**Result 2**

```text
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how
```

### Your observations 4

Did HyDE retrieve more useful evidence, repeat the same passages,
or move the search away from the original question? Identify one claim in the
hypothetical document and check whether the retrieved text actually supports it.

**Your response:**

*HyDE retrieved more useful evidence than the original question. The baseline search returned only consumer opinions and online forum discussions, which did not provide clinical evidence about pregnancy safety. In contrast, the first HyDE result reported a 29% decrease in offspring viability, skeletal development abnormalities at three or more times the maximum recommended human dose, and stated that Zelomax is contraindicated during pregnancy. This supports the hypothetical document’s general claim that Zelomax poses a significant fetal risk and should not be used during pregnancy. However, the retrieved passages do not support many of its specific claims, such as Pregnancy Category D, human post-marketing evidence, particular congenital defects, or the recommendation to use contraception for six months. Therefore, HyDE improved retrieval, but its hypothetical document should not be treated as evidence.*

The distinction to remember is that rewriting produces another question, while
HyDE produces a hypothetical answer. That generated answer is not evidence.
Check whether searching with it retrieved passages that support an answer to
the original question.

## H. Reflection

You have compared four ways to change the input to a retrieval system.

### What You Learned

**Key concepts:**
- **Query Rewriting:** Expressing the same intent in multiple alternative phrasings to increase retrieval coverage
- **Query Expansion:** Enriching queries with synonyms and related terms to bridge vocabulary gaps
- **Query Decomposition:** Breaking complex multi-part questions into simpler, focused sub-questions
- **HyDE (Hypothetical Document Embeddings):** Generating hypothetical answer documents for better semantic matching

**The core insight:**

User queries and documents can often live in different linguistic spaces.

Query transformation techniques bridge this gap by:
- Rephrasing questions to match document language
- Expanding vocabulary to capture domain-specific terminology
- Simplifying complex queries into manageable pieces
- Generating document-style text for better similarity matching

### Optional Discussion Questions

Your written responses belong in the four **Your observations** areas above.
The questions below are for discussion; additional written answers are not required.

1. **Technique selection:** Which transformation technique would work best for different query types? When would you choose query rewriting over HyDE?

2. **Combining techniques:** Could you use multiple transformation techniques together? For example, decompose a complex query, then expand each sub-question with synonyms?

3. **Domain adaptation:** How would you modify these prompts for different domains beyond pharmaceutical? What would change for legal documents, technical manuals, or financial reports?

4. **Trade-offs:** Each technique adds an extra LLM call before retrieval. What evidence would justify the added latency and cost?

5. **Retrieval strategy:** After transforming a query into multiple versions (e.g., 3 rewrites), do you retrieve separately and merge results, or combine the queries first?

6. **Failure modes:** What happens when query transformation goes wrong? How might a poorly decomposed question or an inaccurate hypothetical document hurt retrieval?

7. **Evaluation:** How would you measure whether query transformation is actually improving your RAG system's performance?

Keep these questions in mind as you continue learning about advanced RAG techniques and system design.

## Save Your Work

Save the notebook with your prompts, observations, and code outputs. Before finishing,
check that your explanations refer to the results you actually obtained.